In [49]:
from  selenium import webdriver
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains
import numpy as np
import json

In [50]:
def close_popup(driver):
    try:
        popup_close = WebDriverWait(driver, 2).until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="app"]/div/div/div[7]/div[2]/div[1]/div/div/div[1]/div/div[1]')
            )
        )
        popup_close.click()
        print("Popup closed.")
    except TimeoutException:
        pass

In [51]:
data = []

In [52]:
driver = webdriver.Chrome()

time.sleep(3)

driver.maximize_window()

url = "https://www.magicbricks.com/"

driver.get(url)

# explicit wait
wait = WebDriverWait(driver, 5)

close_popup(driver)

try:

    # First: locate the search bar

    search_bar = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//input[@id='keyword']")
        )
    )

except:

    print("Timeout while locating Search Bar.\n")

else:

    # Second: locate the already written text

    try:

        text = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//div[@class='mb-search__tag-t']")
            )
        )

    except:

        print("No existing text found.")

    else:

        # First click on the existing text

        text.click()

        time.sleep(1)

        # Third: after clicking the text, locate the cross button

        try:

            close_button = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//*[@id="keyword_autoSuggestSelectedDiv"]/div/div[2]')
                )
            )

        except:

            print("No existing close button found.")

        else:

            # Click the cross button

            close_button.click()

            time.sleep(1)

    # Finally: enter the new search term

    search_bar.send_keys("gurgaon")

    time.sleep(1)

# selecting valid response from list
try:
    valid_option = wait.until(
        EC.element_to_be_clickable((By.XPATH,'//*[@id="serachSuggest"]/div[2]'))
    )
except:
    print("Timeout while locating valid search option.\n")
else:
    valid_option.click()
    time.sleep(2)

try:
    enter = wait.until(
        EC.element_to_be_clickable((By.XPATH,'//*[@id="searchFormHolderSection"]/section/div/div[1]/div[3]/div[4]'))
    )
except:
    print("Timeout while locating valid search button.\n")
else:
    enter.click()
    time.sleep(2)

# Third: hover on the first element

try:

    hover_element = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, '//*[@id="propertysrp"]/div[1]/div/div/div[1]/div[3]/span')
        )
    )

except:

    print("Unable to locate the hover element.")

else:

    # Move cursor to the element (hover)
    ActionChains(driver).move_to_element(hover_element).perform()

    time.sleep(1)


# Fourth: after hovering, locate and click "Verified Properties"

try:

    verified_properties = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//span[normalize-space()='Verified Properties']")
        )
    )

except:

    print("Unable to locate Verified Properties.")

else:

    # Click "Verified Properties"
    verified_properties.click()

    time.sleep(1)


# Store the current window

original_window = driver.current_window_handle

# Wait until a new window/tab opens

wait.until(lambda d: len(d.window_handles) > 1)

# Switch to the newly opened window

for window in driver.window_handles:

    if window != original_window:

        driver.switch_to.window(window)

        break

time.sleep(2)

# Third: click the first element

try:

    budget = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[2]/div/div[1]/div')
        )
    )

except:

    print("Unable to locate budget.")

else:

    budget.click()

    time.sleep(1)


# Fourth: click the second element

min_budget = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div/div[1]/div[1]/select')
    )
)

Select(min_budget).select_by_index(9)
time.sleep(2)

max_budget = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div/div[1]/div[3]/select')
    )
)

Select(max_budget).select_by_index(5)
time.sleep(1)

done = wait.until(

    EC.element_to_be_clickable(

        (By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[2]/div/div[2]/div[2]')

    )

)

done.click()

time.sleep(1)

prev_height = driver.execute_script("return document.body.scrollHeight")

while True:
    print(f"previous webpage height {prev_height:,} pixel:")

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    time.sleep(2)

    # calculate h2
    new_height = driver.execute_script("return document.body.scrollHeight")

    if prev_height == new_height:
        break   

    prev_height = new_height 



# Page has reached the bottom

print("Reached the bottom of the page.")

# Scroll up by half of the visible screen

viewport_height = driver.execute_script(

    "return window.innerHeight"

)

driver.execute_script(

    "window.scrollBy(0, arguments[0]);",

    -viewport_height / 3

)

time.sleep(2)

print("Scrolled up by half of the visible screen.")

# Scraping data

# MAIN PROPERTY CONTAINER

rows = driver.find_elements(

    By.CLASS_NAME,

    "mb-srp__card"

)

for row in rows:

    try:
        name = row.find_element(
            By.CLASS_NAME,
            "mb-srp__card--title"
        ).text

    except:
        name = np.nan

    try:
        values = row.find_elements(
        By.CLASS_NAME,
        "mb-srp__card__summary--value"
        )
        furnished = values[0].text

        tenant_preferred = values[2].text 
    except:
        furnished = np.nan    
        tenant_preferred = np.nan                     
                    
    # ----------------------------------------------

    # RENT

    # ----------------------------------------------

    try:

        rent_container = row.find_element(

            By.CLASS_NAME,

            "mb-srp__card__estimate"

        )

        rent = rent_container.find_element(

            By.CLASS_NAME,

            "mb-srp__card__price--amount"

        ).text

    except:

        rent = np.nan

    property ={ "name": name,
               "furnished": furnished,
               "tenant_preferred": tenant_preferred,
               "rent": rent}

    data.append(property)
    

    
# save data as JSON
with open("property_data.json","w") as f:
    json.dump(data,f,indent=4)
print("Data saved successfully!")            


previous webpage height 8,161 pixel:
previous webpage height 15,277 pixel:
previous webpage height 22,298 pixel:
previous webpage height 29,112 pixel:
previous webpage height 35,764 pixel:
previous webpage height 42,608 pixel:
previous webpage height 49,436 pixel:
previous webpage height 56,497 pixel:
previous webpage height 63,603 pixel:
previous webpage height 70,379 pixel:
previous webpage height 77,438 pixel:
previous webpage height 84,534 pixel:
previous webpage height 91,373 pixel:
previous webpage height 98,514 pixel:
previous webpage height 105,417 pixel:
previous webpage height 112,532 pixel:
previous webpage height 119,392 pixel:
previous webpage height 126,439 pixel:
previous webpage height 133,412 pixel:
previous webpage height 140,406 pixel:
previous webpage height 147,345 pixel:
previous webpage height 154,259 pixel:
previous webpage height 161,158 pixel:
previous webpage height 168,103 pixel:
previous webpage height 174,941 pixel:
previous webpage height 181,796 pixel:
p

In [53]:
len(data)

893